# GIL у Python: багатопоточність, I/O та CPU-bound задачі

Цей ноутбук показує, як Global Interpreter Lock впливає на потоки в Python. Йдемо від простого прикладу з I/O до CPU-bound роботи, перемикання потоків, порядку виконання та експериментів із C-розширеннями.


## Підготовка

Спочатку імпортуємо модулі, які потрібні для всіх прикладів. Тут немає спеціальних бібліотек: усе працює на стандартній бібліотеці Python.


In [ ]:
import os
import platform
import sys
import sysconfig
import threading
import time
import zlib
from concurrent.futures import ThreadPoolExecutor

## I/O-bound задача: потоки можуть перекриватися

У цьому прикладі кожен потік просто “чекає” через `time.sleep`. Поки один потік чекає, інші теж можуть виконувати свою частину роботи. Тому загальний час ближчий до 2 секунд, а не до 8.


In [ ]:
def run_io_task(name: str, delay: float = 2.0) -> None:
    print(f'{name} почав I/O операцію')
    time.sleep(delay)
    print(f'{name} закінчив I/O операцію')


threads = [threading.Thread(target=run_io_task, args=(f'Потік-{i}',)) for i in range(4)]

start = time.perf_counter()

for thread in threads:
    thread.start()

for thread in threads:
    thread.join()

print(f'Загальний час: {time.perf_counter() - start:.2f}s')

На що звернути увагу: тут GIL не заважає, бо задача майже не "рахує" Python-код. Для I/O-bound задач потоки часто дають практичну користь.


Запустивши в терміналі `py-spy top -- python path_to_this_script.py`, можете прослідкувати за внутрішніми метриками та участю GIL у виконанні коду.

## CPU-bound задача: потоки не дають очікуваного прискорення

Тепер задача постійно "рахує" в Python. Потоки стартують "паралельно", але GIL дозволяє виконувати Python-байткод лише одному потоку в конкретний момент часу.


In [ ]:
def cpu_task(n: int) -> int:
    total = 0
    for value in range(n):
        total += value * value
    return total


def run_cpu_threads(worker_count: int, n: int) -> float:
    results = [0] * worker_count
    threads = []

    start = time.perf_counter()

    for index in range(worker_count):
        thread = threading.Thread(target=lambda i=index: results.__setitem__(i, cpu_task(n)))
        threads.append(thread)
        thread.start()

    for thread in threads:
        thread.join()

    return time.perf_counter() - start


n = 3_000_000
worker_count = min(4, os.cpu_count() or 1)

one_thread = run_cpu_threads(1, n)
many_threads = run_cpu_threads(worker_count, n)

print(f'1 thread       : {one_thread:.3f}s')
print(f'{worker_count} threads: {many_threads:.3f}s')
print(f'Speedup        : {one_thread / many_threads:.2f}x')

Чому це важливо: для важких обчислень у чистому Python звичайні потоки часто не масштабуються на кілька ядер. Для таких задач зазвичай дивляться у бік процесів, NumPy/C-розширень або free-threaded Python.


## Як часто Python перемикає потоки

`sys.setswitchinterval()` задає приблизний інтервал, після якого інтерпретатор може віддати керування іншому потоку. Це не робить CPU-bound код паралельним, але впливає на те, наскільки “чесно” потоки ділять час.


In [ ]:
def cpu_spin(iterations: int) -> int:
    total = 0
    for value in range(iterations):
        total += value
    return total


def timed_worker(name: str, iterations: int) -> tuple[str, float]:
    start = time.perf_counter()
    cpu_spin(iterations)
    elapsed = time.perf_counter() - start
    return name, elapsed


def run_switch_experiment(label: str, switch_interval: float, iterations: int) -> None:
    previous_interval = sys.getswitchinterval()
    sys.setswitchinterval(switch_interval)

    try:
        start = time.perf_counter()
        with ThreadPoolExecutor(max_workers=2) as executor:
            results = list(
                executor.map(
                    timed_worker,
                    ['T0', 'T1'],
                    [iterations, iterations],
                )
            )
        total = time.perf_counter() - start
    finally:
        sys.setswitchinterval(previous_interval)

    results.sort()
    times = [elapsed for _, elapsed in results]
    fairness_gap = abs(times[0] - times[1])
    fairness_ratio = min(times) / max(times)

    print(f'\n=== {label} ===')
    print(f'switch interval = {switch_interval}')
    for name, elapsed in results:
        print(f'{name}: elapsed={elapsed:.4f}s')
    print(f'Total wall time : {total:.4f}s')
    print(f'Fairness gap    : {fairness_gap:.4f}s')
    print(f'Fairness ratio  : {fairness_ratio:.4f}')


run_switch_experiment('Frequent switching', 0.001, 3_000_000)
run_switch_experiment('Less frequent switching', 0.05, 3_000_000)

Що тут відбувається: ми запускаємо два однакові CPU-bound потоки і дивимось, як близько їхній час виконання один до одного. Менший інтервал зазвичай дає частіші перемикання, але сам по собі не створює справжній паралелізм для Python-коду.


## Порядок виконання потоків не гарантований

Навіть якщо потоки стартують майже одночасно, порядок їх завершення або входу в критичну секцію може змінюватися. Планувальник ОС і GIL разом роблять цей порядок недетермінованим.


In [ ]:
def order_worker(
    name: str,
    barrier: threading.Barrier,
    lock: threading.Lock,
    order: list[str],
) -> None:
    barrier.wait()
    time.sleep(0.001)

    with lock:
        order.append(name)


def run_order_once(thread_count: int = 4) -> list[str]:
    barrier = threading.Barrier(thread_count)
    lock = threading.Lock()
    order: list[str] = []

    threads = [
        threading.Thread(
            target=order_worker,
            args=(f'T{i}', barrier, lock, order),
        )
        for i in range(thread_count)
    ]

    for thread in threads:
        thread.start()

    for thread in threads:
        thread.join()

    return order


for run_number in range(1, 6):
    print(f'Run {run_number}: {run_order_once()}')

На що звернути увагу: `Lock` захищає список від одночасного запису, але не фіксує порядок потоків. Порядок потоків не керується на рівні Python.


## C-розширення можуть тимчасово відпускати GIL

Не весь код виконується як Python-байткод. Деякі C-розширення можуть відпускати GIL на час важкої CPU роботи, і тоді потоки іноді справді працюють паралельно.


In [ ]:
payload = b'Python GIL demo ' * 1_000_000


def compress_many(times: int) -> None:
    for _ in range(times):
        zlib.compress(payload)


def run_single_compression() -> float:
    start = time.perf_counter()
    compress_many(4)
    return time.perf_counter() - start


def run_threaded_compression() -> float:
    threads = [threading.Thread(target=compress_many, args=(2,)) for _ in range(2)]

    start = time.perf_counter()

    for thread in threads:
        thread.start()

    for thread in threads:
        thread.join()

    return time.perf_counter() - start


single_time = run_single_compression()
threaded_time = run_threaded_compression()

print(f'Single thread: {single_time:.4f}s')
print(f'Two threads  : {threaded_time:.4f}s')
print(f'Ratio        : {single_time / threaded_time:.2f}x')

Результат може відрізнятися на різних машинах. Головна ідея: GIL найбільше обмежує Python-код, але робота всередині C-бібліотек може поводитися інакше.


## Free-threaded Python: як перевіряти майбутнє без GIL

Останній приклад повторює CPU-bound перевірку, але корисний для запуску у free-threaded збірці Python. У звичайному CPython прискорення від потоків для такого коду зазвичай не буде взагалі або навіть матиме накладні витрати.


In [ ]:
def run_free_threading_check(n: int = 3_000_000) -> None:
    worker_count = min(4, os.cpu_count() or 1)

    print(f'Python: {sys.version}')
    print('Implementation:', platform.python_implementation())
    print('Has sys._is_gil_enabled:', hasattr(sys, '_is_gil_enabled'))

    if hasattr(sys, '_is_gil_enabled'):
        print('GIL enabled:', sys._is_gil_enabled())

    print('Py_GIL_DISABLED:', sysconfig.get_config_var('Py_GIL_DISABLED'))
    print(f'Threads: {worker_count}')

    one_thread = run_cpu_threads(1, n)
    many_threads = run_cpu_threads(worker_count, n)

    print(f'1 thread       : {one_thread:.3f}s')
    print(f'{worker_count} threads: {many_threads:.3f}s')
    print(f'Speedup        : {one_thread / many_threads:.2f}x')


run_free_threading_check()

Для окремого запуску поза ноутбуком можна порівняти звичайний Python і free-threaded збірку, якщо вона встановлена:

```bash
python path_to_this_script.py
python3.14t -X gil=0 path_to_this_script.py
```

Тут важливо дивитися не лише на сам факт вимкнення GIL, а й на реальний `Speedup` для конкретної задачі.


## Підсумок

GIL не означає, що потоки в Python “не працюють”. Вони добре підходять для I/O-bound задач, але погано масштабують чистий Python CPU-bound код. Для обчислень варто обирати інший інструмент: процеси, C/NumPy-операції або free-threaded Python там, де це доречно.
